# FL/OD with Green-Input Duty Cycle

This notebook parses a `.out` serial-command log and a FL/OD `.csv` file,
then plots FL/OD (orange) overlaid with the duty cycle of the **green (`g`)
light input** (green shaded area) for each well.


In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as ticker
from matplotlib.ticker import FuncFormatter

%matplotlib inline
plt.rcParams.update({"figure.dpi": 150})

# ── File paths ───────────────────────────────────────────────────────────────
OUT_FILE = "Template_b.out"   # serial command log
CSV_FILE = "Datafile/fl_od_all_wells.csv"  # FL/OD data

# ── Rolling window for duty-cycle calculation (minutes) ───────────────────────
WINDOW_MIN = 10
LIGHTS_ON_MIN    = 8   # minutes lights are on per cycle
CYCLE_PERIOD_MIN = 10  # total cycle period in minutes

# ── Well order encoded in the .out file (32 positions) ───────────────────────
CTRL_WELLS = [
    'A1','B1','C1','D1','E1','F1','G1','H1',
    'H3','G3','F3','E3','D3','C3','B3','A3',
    'A4','B4','C4','D4','E4','F4','G4','H4',
    'H6','G6','F6','E6','D6','C6','B6','A6'
]

In [ ]:
def parse_out(path):
    """
    Parse the serial command log.

    Returns
    -------
    list of (time_min, cmd) tuples
        time_min : float  – timestamp converted to minutes
        cmd      : str    – 32-character command string (one char per well)
    """
    events = []
    with open(path) as f:
        content = f.read()
    for m in re.finditer(r'([\d.]+): Writing ([a-z]{32})', content):
        events.append((float(m.group(1)) / 60.0, m.group(2)))
    return events


events = parse_out(OUT_FILE)
print(f"Parsed {len(events)} events")
print(f"Time range: {events[0][0]:.1f} – {events[-1][0]:.1f} min")
print(f"\nFirst 5 events:")
for t, cmd in events[:5]:
    print(f"  {t:8.2f} min  |  {cmd}")


In [ ]:
df = pd.read_csv(CSV_FILE)
wells = [c for c in df.columns if c != "Time_min"]

print(f"Wells in CSV: {wells}")
print(f"Time range:  {df.Time_min.min():.1f} – {df.Time_min.max():.1f} min")
print(f"Rows: {len(df)}")
df.head()


## Duty-Cycle Calculation

For each FL/OD time point the duty cycle is the fraction of the preceding
`WINDOW_MIN` minutes during which the `g` input was active for that well.


In [ ]:
def duty_cycle(events, well, time_points,
               window_min=WINDOW_MIN,
               lights_on_min=LIGHTS_ON_MIN,
               cycle_period_min=CYCLE_PERIOD_MIN):
    """
    Compute rolling duty cycle of the 'g' input for *well*.

    Duty cycle is normalised by the *maximum possible* 'g' time inside the
    rolling window, not the full window duration.  Since the lights are only
    on for `lights_on_min` out of every `cycle_period_min` minutes, the max
    possible 'g' time is window_min * (lights_on_min / cycle_period_min).

    Example: lights on 8/10 min, window=20 min → max_g=16 min.
             If 8 min of 'g' observed → 50% duty cycle.

    Parameters
    ----------
    events           : list of (time_min, cmd) from parse_out()
    well             : str   – well name, e.g. 'F6'
    time_points      : array-like of floats (minutes)
    window_min       : float – rolling window width in minutes
    lights_on_min    : float – minutes lights are on per cycle (default 8)
    cycle_period_min : float – total cycle period in minutes (default 10)

    Returns
    -------
    np.ndarray, same length as time_points, values in [0, 100] %
    """
    idx        = CTRL_WELLS.index(well)
    max_g_time = window_min * (lights_on_min / cycle_period_min)
    dc         = np.zeros(len(time_points))

    for j, t in enumerate(time_points):
        t0      = t - window_min
        total_g = 0.0
        for i in range(1, len(events)):
            t_prev, cmd_prev = events[i - 1]
            t_curr, _        = events[i]
            if cmd_prev[idx] != 'g':
                continue
            seg_start = max(t_prev, t0)
            seg_end   = min(t_curr, t)
            if seg_end > seg_start:
                total_g += seg_end - seg_start
        dc[j] = (total_g / max_g_time) * 100.0

    return dc


# Quick sanity check for the first well
_dc = duty_cycle(events, wells[0], df.Time_min.values)
print(f"Duty cycle for {wells[0]}: min={_dc.min():.1f}%  max={_dc.max():.1f}%")
print(f"(normalised by max possible g time = "
      f"{WINDOW_MIN * LIGHTS_ON_MIN / CYCLE_PERIOD_MIN:.1f} min "
      f"per {WINDOW_MIN}-min window)")


## Plot

In [ ]:
# ── Colour palette ────────────────────────────────────────────────────────────
ORANGE     = '#E07B39'
GREEN_FILL = '#5DB85D'
GREEN_LINE = '#2E7D2E'
GREEN_DOT  = '#2E7D2E'

# ── PID groups (matches controller's pid1..pid10) ───────────────────────────
# Used only to order/label the grid below; duty_cycle() itself is well-agnostic.
pid_groups = {
    'pid1_sp1':  ['C1','D1','E1'],
    'pid2_sp1':  ['F1','G1','H1'],
    'pid3_sp1':  ['C3','C4','C6'],
    'pid4_sp1':  ['D3','D4','D6'],
    'pid1_sp2':  ['E3','E4','E6'],
    'pid2_sp2':  ['F3','F4','F6'],
    'pid3_sp2':  ['G3','G4','G6'],
    'pid4_sp2':  ['H3','H4','H6'],
}

# Order wells group-by-group (falls back gracefully if the CSV is missing
# a well — e.g. only some PID groups were measured this run)
ordered_wells = [w for g in pid_groups.values() for w in g if w in wells]

ncols = 3
nrows = len(pid_groups)
fig, axes = plt.subplots(nrows, ncols,
                         figsize=(4.5 * ncols, 3 * nrows),
                         constrained_layout=True)
axes = np.atleast_2d(axes)

for row, (group_name, group_wells) in enumerate(pid_groups.items()):
    for col in range(ncols):
        ax = axes[row, col]

        if col >= len(group_wells) or group_wells[col] not in wells:
            ax.axis('off')
            continue

        well = group_wells[col]
        t_data = df['Time_min'].values
        fl_od  = df[well].values
        dc     = duty_cycle(events, well, t_data)

        # ── Right axis: duty cycle ────────────────────────────────────────
        ax2 = ax.twinx()

        ax2.fill_between(t_data, dc, 0,
                         color=GREEN_FILL, alpha=0.25, linewidth=0, zorder=1)

        # White vertical stripes for a hatched look
        for stripe_t in np.arange(t_data[0], t_data[-1], 4):
            ax2.axvline(stripe_t, color='white', linewidth=0.5,
                        alpha=0.6, zorder=2)

        ax2.plot(t_data, dc,
                 color=GREEN_LINE, linewidth=1.6, linestyle='--', zorder=3)
        ax2.scatter(t_data, dc,
                    color=GREEN_DOT, s=18, zorder=4)

        ax2.set_ylim(0, 100)
        ax2.set_yticks([0, 25, 50, 75, 100])
        ax2.tick_params(axis='y', colors=GREEN_LINE, labelsize=9)
        ax2.spines['right'].set_color(GREEN_LINE)
        if col == ncols - 1:
            ax2.set_ylabel('Duty Cycle (%)', color=GREEN_LINE, fontsize=10)
        else:
            ax2.set_yticklabels([])

        # ── Left axis: FL/OD ───────────────────────────────────────────────
        ax.axhline(15000, color=ORANGE, linewidth=1.5,
                   linestyle='--', zorder=5, alpha=0.85)
        ax.plot(t_data, fl_od, color=ORANGE, linewidth=2.2, zorder=6)

        ax.set_ylim(bottom=0)
        ax.set_xlim(0, 960)
        ax.tick_params(labelsize=9)
        ax.yaxis.set_major_formatter(
            plt.FuncFormatter(
                lambda x, _: f'{x/1000:.0f}k' if x >= 1000 else f'{x:.0f}'
            )
        )
        if row == nrows - 1:
            ax.set_xlabel('Time (mins)', fontsize=10)
        if col == 0:
            ax.set_ylabel('FL/OD (a.u.)', fontsize=10)

        ax.set_title(well, fontsize=11, fontweight='bold', pad=6)
        ax.set_zorder(ax2.get_zorder() + 1)
        ax.patch.set_visible(False)

    # Row label identifying the PID group, placed left of the first column
    axes[row, 0].annotate(
        group_name, xy=(0, 0.5), xycoords='axes fraction',
        xytext=(-58, 0), textcoords='offset points',
        ha='center', va='center', rotation=90,
        fontsize=11, fontweight='bold'
    )

# ── Legend ────────────────────────────────────────────────────────────────────
fl_patch = mpatches.Patch(color=ORANGE,     label='FL/OD')
dc_patch = mpatches.Patch(color=GREEN_FILL, alpha=0.6, label="'g' duty cycle")
fig.legend(handles=[fl_patch, dc_patch],
           loc='upper right', frameon=False, fontsize=9)

fig.suptitle('Timing Diagram — all PID groups',
             fontsize=14, fontweight='bold', y=1.02)
plt.show()

save_path = "fl_od_with_duty_cycle.png"
fig.savefig(save_path, dpi=180, bbox_inches='tight', facecolor='white')
print(f"Figure saved to {save_path}")


In [ ]:
# ── Choose which well to plot ─────────────────────────────────────────────────
SINGLE_WELL = 'H3'  # change this to any well present in `wells`
assert SINGLE_WELL in wells, f"{SINGLE_WELL} not found in CSV wells: {wells}"

idx = CTRL_WELLS.index(SINGLE_WELL)

# ── Bin settings ───────────────────────────────────────────────────────────────
interval_length = CYCLE_PERIOD_MIN  # 10 min per bar
max_g_per_bin   = LIGHTS_ON_MIN     # 8 min — max possible green time per bin
time_limit_min  = 10 * 60           # 10 hours, in minutes (matches x-axis limit)
bar_fraction    = 0.7               # fraction of bin width drawn as the bar (<1 leaves a gap)

num_bins = int(np.ceil(time_limit_min / interval_length))

# ── Compute duty cycle per bin: % of the 8-min "on" budget that was green ─────
bin_edges = [i * interval_length for i in range(num_bins + 1)]
duty_vals = []
for i in range(num_bins):
    t0, t1 = bin_edges[i], bin_edges[i + 1]
    green_time = 0.0
    for j in range(1, len(events)):
        t_prev, cmd_prev = events[j - 1]
        t_curr, _        = events[j]
        if cmd_prev[idx] != 'g':
            continue
        seg_start = max(t_prev, t0)
        seg_end   = min(t_curr, t1)
        if seg_end > seg_start:
            green_time += seg_end - seg_start
    duty_vals.append(min((green_time / max_g_per_bin) * 100.0, 100.0))

duty_times_hr = [(bin_edges[i] + bin_edges[i + 1]) / 2 / 60.0 for i in range(num_bins)]  # bin centers, in hours
bar_width_hr  = (interval_length / 60.0) * bar_fraction

# ── FL/OD data (still continuous, in hours) ────────────────────────────────────
t_data_min = df['Time_min'].values
t_data     = t_data_min / 60.0
fl_od      = df[SINGLE_WELL].values

time_limit = 10  # x-axis upper limit (hours)

fig, ax1 = plt.subplots(figsize=(3, 2), dpi=300)

# Style
plt.rcParams.update({
    'font.family': 'Arial',
    'font.size': 7,
    'axes.labelsize': 7,
    'axes.titlesize': 6
})

flod_color = 'black'
duty_color = '#74B867'
st_pt_1_color = '#E07B39'
tick_font_size = 7
lwd_model = 1.5
alp = 0.5

# Plot FL/OD signal
ax1.plot(t_data, fl_od, color=st_pt_1_color, linewidth=lwd_model, label=f'{SINGLE_WELL}')
ax1.set_xlabel('Time (h)', labelpad=2)
ax1.set_ylabel('FL/OD (a.u.)', color=flod_color, labelpad=2)
ax1.tick_params(axis='y', labelcolor=flod_color, pad=2)
ax1.set_xlim(0, time_limit)
ax1.set_ylim(0, 22000)
ax1.minorticks_on()
ax1.xaxis.set_major_locator(ticker.MultipleLocator(2))
ax1.xaxis.set_minor_locator(ticker.MultipleLocator(1))
ax1.yaxis.set_minor_locator(ticker.MultipleLocator(5000))
ax1.yaxis.set_minor_locator(ticker.MultipleLocator(2500))

ax1.axhline(y=18500, linestyle='--', color=st_pt_1_color, linewidth=lwd_model)

def thousands_formatter(x, pos):
    return f'{int(x/1000)}k'

ax1.yaxis.set_major_formatter(FuncFormatter(thousands_formatter))

# Secondary y-axis: Duty Cycle, as bars per interval
ax2 = ax1.twinx()

ax2.bar(duty_times_hr, duty_vals, width=bar_width_hr,
        color=duty_color, alpha=alp, edgecolor='none')
ax2.plot(duty_times_hr, duty_vals, color=duty_color, linestyle='--', linewidth=lwd_model)

ax2.set_ylabel("Duty Cycle (%)", color=duty_color, labelpad=2)
ax2.tick_params(axis='y', labelcolor=duty_color, pad=2,width=0.5)
ax2.set_ylim(0, 99)
ax2.yaxis.set_minor_locator(ticker.MultipleLocator(10))

# Spine styling
spine_width = 0.5
for ax in [ax1, ax2]:
    for spine in ['top', 'right', 'bottom', 'left']:
        ax.spines[spine].set_visible(True)
        ax.spines[spine].set_linewidth(spine_width)
    ax.tick_params(axis='both', which='major', labelsize=tick_font_size, pad=2,width=0.5)

# plt.tight_layout()
# fig.subplots_adjust(
#     left=0.18,
#     right=0.76,
#     bottom=0.18,
#     top=0.82
# )
plt.show()

fig.savefig(f'timing_diagram_{SINGLE_WELL}.svg', format='svg', dpi=300, bbox_inches='tight', transparent=True)